# **Testing API**

API URL to use https://west.albion-online-data.com

Endpoint
price in json
https://west.albion-online-data.com/api/v2/stats/prices/T4_HIDE.json?locations=Lymhurst,Martlock,Bridgewatch,thetford,FortSterling,Caerleon

### **Coleta de dados**

In [214]:
import requests
import pandas as pd
import re
from IPython.display import display


def get_albion_prices(item_id, locations=None):
    """Busca os preços de um item no endpoint da Albion Online."""
    item_search = ""
    for i in item_id:
        item_search = item_search + f'{i},'
    url = f"https://west.albion-online-data.com/api/v2/stats/prices/{item_search}.json"
    params = {"locations": ",".join(locations)} if locations else None

    print(url)

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.json()


def format_timedelta(td):
    if pd.isna(td):
        return None
    total_seconds = int(td.total_seconds())
    if total_seconds >= 86400:
        return ""
    value = f"{total_seconds // 3600:02d}:{(total_seconds % 3600) // 60:02d}:{total_seconds % 60:02d}"
    return re.sub(r"^(\d{2}):(\d{2}):(\d{2})$", r"\1:\2:\3", value)


def safe_dt(value):
    if value is None or value == 0 or pd.isna(value):
        return pd.NaT
    try:
        return pd.to_datetime(value, utc=True, errors="coerce")
    except Exception:
        return pd.NaT


def add_time_since_update(df, date_col):
    df = df.copy()
    df[date_col] = df[date_col].apply(safe_dt)
    df["time_since_update"] = df.apply(
        lambda row: (
            pd.Timestamp.utcnow() - row[date_col]
        ) if pd.notna(row[date_col]) else pd.NaT,
        axis=1,
    )
    df["time_since_update"] = df["time_since_update"].apply(format_timedelta)
    return df


def get_value_for_item(item_name, mapping):
    if item_name in mapping:
        return mapping[item_name]

    for key, value in mapping.items():
        if item_name.startswith(key.split("_LEVEL")[0]):
            return value

    return 0


def to_refined_item(item_name):
    return re.sub(r"HIDE", "LEATHER", item_name)


In [215]:
item_raw = ["T4_HIDE", "T4_HIDE_LEVEL1@1", "T4_HIDE_LEVEL2@2", "T4_HIDE_LEVEL3@3"]
item_raw2 = ["T3_LEATHER"]
item_refined = ["T4_LEATHER", "T4_LEATHER_LEVEL1@1", "T4_LEATHER_LEVEL2@2", "T4_LEATHER_LEVEL3@3"]
item_value = [16, 32, 64, 128]
cities = ["Lymhurst", "Martlock", "Bridgewatch", "Thetford", "FortSterling"]

value_by_item = dict(zip(item_raw, item_value))
value_by_refined = dict(zip(item_refined, item_value))

In [227]:
#Coleta e formatação de dados
prices_raw = get_albion_prices(item_raw, cities)
prices_raw = [item for item in prices_raw if item.get("quality") == 1]

prices_raw2 = get_albion_prices(item_raw2, cities)
prices_raw2 = [item for item in prices_raw2 if item.get("quality") == 1]

prices_refined = get_albion_prices(item_refined, cities)
prices_refined = [item for item in prices_refined if item.get("quality") == 1]


raw_table = pd.DataFrame([
    {
        "raw": item.get("item_id"),
        "city": item.get("city"),
        "sell_price_min": item.get("sell_price_min"),
        "sell_price_min_date": item.get("sell_price_min_date")
    }
    for item in prices_raw
])

raw2_table = pd.DataFrame([
    {
        "raw": item.get("item_id"),
        "city": item.get("city"),
        "sell_price_min": item.get("sell_price_min"),
        "sell_price_min_date": item.get("sell_price_min_date")
    }
    for item in prices_raw2
])

refined_table = pd.DataFrame([
    {
        "refined": item.get("item_id"),
        "city": item.get("city"),
        "buy_price_max": item.get("buy_price_max"),
        "buy_price_max_date": item.get("buy_price_max_date"),
        "sell_price_min": item.get("sell_price_min"),
        "sell_price_min_date": item.get("sell_price_min_date")
    }
    for item in prices_refined
])

raw_table = add_time_since_update(raw_table, "sell_price_min_date")
raw2_table = add_time_since_update(raw2_table, "sell_price_min_date")
refined_table = add_time_since_update(refined_table, "buy_price_max_date")
refined_table = add_time_since_update(refined_table, "sell_price_min_date")

https://west.albion-online-data.com/api/v2/stats/prices/T4_HIDE,T4_HIDE_LEVEL1@1,T4_HIDE_LEVEL2@2,T4_HIDE_LEVEL3@3,.json
https://west.albion-online-data.com/api/v2/stats/prices/T3_LEATHER,.json
https://west.albion-online-data.com/api/v2/stats/prices/T4_LEATHER,T4_LEATHER_LEVEL1@1,T4_LEATHER_LEVEL2@2,T4_LEATHER_LEVEL3@3,.json


In [ ]:
raw_table

In [ ]:
raw2_table

In [ ]:
refined_table

### **Analise simples**

In [ ]:
cost = pd.DataFrame()
cost['city'] = raw_table['city']
cost['pelego_amount'] = 2
cost['pelego_raw'] = raw_table['sell_price_min']
cost['couro_amount'] = 1
cost['couro_raw'] = raw2_table['sell_price_min']
cost['retorno'] = 0.438
cost['fee'] = round(get_value_for_item(item_raw[0], value_by_item) * 1.05, 0)
cost['consumo_efetivo'] = (cost['pelego_amount'] * cost['pelego_raw'] + cost['couro_amount'] * cost['couro_raw']) * (1 - cost['retorno']) + cost['fee']
cost['sell_tax'] = 0.04
cost['sell_tax_order'] = 0.025
cost['sell_price_min'] = refined_table['buy_price_max']
cost['sell_price_max'] = refined_table['sell_price_min']
cost['profit'] = cost['sell_price_min'] - cost['consumo_efetivo'] - (cost['sell_price_min'] * cost['sell_tax'])
cost['profit_max'] = cost['sell_price_max'] - cost['consumo_efetivo'] - (cost['sell_price_max'] * cost['sell_tax']) + cost['sell_price_max'] * cost['sell_tax_order']
cost['profit_percentage'] = round((cost['profit'] / cost['consumo_efetivo']) * 100, 2)
cost['profit_percentage_max'] = round((cost['profit_max'] / cost['consumo_efetivo']) * 100, 2)

In [125]:
cost

,city,pelego_amount,pelego_raw,couro_amount,couro_raw,retorno,fee,consumo_efetivo,sell_tax,sell_tax_order,sell_price_min,sell_price_max,profit,profit_max,profit_percentage,profit_percentage_max
0,Bridgewatch,2,120,1,259,0.438,17.0,297.438,0.04,0.025,275,284,-33.438,-17.698,-11.24,-5.95
1,Fort Sterling,2,139,1,244,0.438,17.0,310.364,0.04,0.025,275,308,-46.364,-6.984,-14.94,-2.25
2,Lymhurst,2,131,1,237,0.438,17.0,297.438,0.04,0.025,287,307,-21.918,4.957,-7.37,1.67
3,Martlock,2,159,1,240,0.438,17.0,330.596,0.04,0.025,244,251,-96.356,-83.361,-29.15,-25.22
4,Thetford,2,204,1,238,0.438,17.0,380.052,0.04,0.025,233,267,-156.372,-117.057,-41.14,-30.80


### **Analise Combinatoria**

In [ ]:
resource_name = to_refined_item(item_raw[0])

# Primeira tabela: combina o item bruto com cada cidade de origem
table1 = pd.DataFrame({
    "resource1": raw_table["raw"],
    "resource2": item_raw2[0],
    "city_raw1": raw_table["city"],
    "pelego_amount": 2,
    "pelego_raw": raw_table["sell_price_min"],
})

# Segunda tabela: combina o item secundário com cada cidade de origem do segundo recurso
table2 = pd.DataFrame({
    "city_raw2": raw2_table["city"],
    "resource2": item_raw2[0],
    "couro_amount": 1,
    "couro_raw": raw2_table["sell_price_min"],
})

# Terceira tabela: combina os itens refinados com as cidades de venda
table3 = pd.DataFrame({
    "resource1": refined_table.apply(lambda x: to_refined_item(x["refined"]), axis=1),
    "resource_end": refined_table["refined"],
    "city_sold": refined_table["city"],
    "couro_refined": 1,
    "couro_raw_instant": refined_table["buy_price_max"],
    "couro_raw_order": refined_table["sell_price_min"],
})

# Produto cartesiano entre as tabelas 1 e 2 usando o recurso comum
base = table1.merge(table2, on=["resource2"], how="outer")

# Join final pelo recurso refinado equivalente
# Isso faz a correspondência entre o item bruto e o item refinado
merged = base.merge(table3, on=["resource1"], how="left")

merged["retorno"] = 0.438
merged["fee"] = round(get_value_for_item(resource_name, value_by_refined) * 1.05, 0)
merged["consumo_efetivo"] = (merged["pelego_amount"] * merged["pelego_raw"] + merged["couro_amount"] * merged["couro_raw"]) * (1 - merged["retorno"]) + merged["fee"]
merged["sell_tax"] = 0.04
merged["sell_tax_order"] = 0.025
merged["profit_instant"] = merged["couro_raw_instant"] - merged["consumo_efetivo"] - (merged["couro_raw_instant"] * merged["sell_tax"])
merged["profit_order"] = merged["couro_raw_order"] - merged["consumo_efetivo"] - (merged["couro_raw_order"] * merged["sell_tax"]) + merged["couro_raw_order"] * merged["sell_tax_order"]
merged["profit_percentage_instant"] = round((merged["profit_instant"] / merged["consumo_efetivo"]) * 100, 2)
merged["profit_percentage_order"] = round((merged["profit_order"] / merged["consumo_efetivo"]) * 100, 2)
merged["best_profit_percentage"] = merged[["profit_percentage_instant", "profit_percentage_order"]].max(axis=1)


df_instant = merged[merged["profit_instant"] > 0]
df_order = merged[merged["profit_order"] > 0]

df_result = pd.concat([df_instant, df_order]).drop_duplicates().reset_index(drop=True).sort_values("best_profit_percentage", ascending=False)
df_result = df_result[["resource_end", "city_raw1", "city_raw2", "city_sold", "pelego_amount", "pelego_raw", "couro_amount", "couro_raw", "couro_raw_instant", "couro_raw_order", "consumo_efetivo", "profit_instant", "profit_order", "profit_percentage_instant", "profit_percentage_order", "best_profit_percentage"]

In [238]:
df_merged

,resource1,resource2,city_raw1,pelego_amount,pelego_raw,city_raw2,couro_amount,couro_raw,resource_end,city_sold,...,retorno,fee,consumo_efetivo,sell_tax,sell_tax_order,profit_instant,profit_order,profit_percentage_instant,profit_percentage_order,best_profit_percentage
0,T4_HIDE,T3_LEATHER,Bridgewatch,2.0,120.0,Bridgewatch,1.0,219.0,NaN,NaN,...,0.438,17.0,274.958,0.04,0.025,NaN,NaN,NaN,NaN,NaN
1,T4_HIDE,T3_LEATHER,Bridgewatch,2.0,120.0,Fort Sterling,1.0,244.0,NaN,NaN,...,0.438,17.0,289.008,0.04,0.025,NaN,NaN,NaN,NaN,NaN
2,T4_HIDE,T3_LEATHER,Bridgewatch,2.0,120.0,Lymhurst,1.0,221.0,NaN,NaN,...,0.438,17.0,276.082,0.04,0.025,NaN,NaN,NaN,NaN,NaN
3,T4_HIDE,T3_LEATHER,Bridgewatch,2.0,120.0,Martlock,1.0,244.0,NaN,NaN,...,0.438,17.0,289.008,0.04,0.025,NaN,NaN,NaN,NaN,NaN
4,T4_HIDE,T3_LEATHER,Bridgewatch,2.0,120.0,Thetford,1.0,238.0,NaN,NaN,...,0.438,17.0,285.636,0.04,0.025,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,T4_LEATHER_LEVEL3@3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T4_LEATHER_LEVEL3@3,Bridgewatch,...,0.438,17.0,NaN,0.04,0.025,NaN,NaN,NaN,NaN,NaN
116,T4_LEATHER_LEVEL3@3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T4_LEATHER_LEVEL3@3,Fort Sterling,...,0.438,17.0,NaN,0.04,0.025,NaN,NaN,NaN,NaN,NaN
117,T4_LEATHER_LEVEL3@3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T4_LEATHER_LEVEL3@3,Lymhurst,...,0.438,17.0,NaN,0.04,0.025,NaN,NaN,NaN,NaN,NaN
118,T4_LEATHER_LEVEL3@3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T4_LEATHER_LEVEL3@3,Martlock,...,0.438,17.0,NaN,0.04,0.025,NaN,NaN,NaN,NaN,NaN
